# Статистический анализ точности системы лазерного наведения с БПЛА

Настоящий анализ посвящён количественной оценке точности системы лазерного наведения, установленной на квадрокоптере со стабилизированным подвесом. Цель исследования — установить, какие метеорологические и навигационные факторы определяют точность наведения, и построить прогностическую модель для планирования полётов.

**Основная метрика качества** — радиальная ошибка $R = \sqrt{r_x^2 + r_y^2}$ (мм), определяемая как горизонтальное расстояние от центра лазерного пятна до реперной точки на поверхности земли.

### Характеристика выборки

| Параметр | Значение |
|---|---|
| Период проведения испытаний | апрель — октябрь 2025 г. |
| Число лётных дней | 59 |
| Контрольные точки | 5 (P1–P5) |
| Высоты зависания | $H \in \{3,\; 5,\; 10,\; 15,\; 20\}$ м |
| Повторений на каждую комбинацию (точка × высота) | 3 |
| Общее число измерений | $59 \times 5 \times 5 \times 3 = 4\,425$ |

Для каждого измерения регистрировались: момент времени, идентификатор точки, высота зависания $H$, номер повторения, локальная скорость ветра $V$ (м/с), интенсивность порывов, направление ветра, облачность (%), число видимых спутников GNSS, а также компоненты отклонения $r_x$, $r_y$ и радиальная ошибка $R$.

In [537]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "output"
JSON_PATH = BASE_DIR / "balanced_selection" / "selected_days_2024-2025.json"

H_ORDER = [3, 5, 10, 15, 20]
H_CAT_ORDER = ["3 м", "5 м", "10 м", "15 м", "20 м"]
ALPHA = 0.05

COL_MAP = {
    "Время": "time", "Точка": "point", "H (м)": "H", "Полёт": "flight",
    "V (м/с)": "V", "Порывы, откр. источник (м/с)": "gusts",
    "Направление ветра": "wind_dir", "Облачность (%)": "cloud",
    "Спутники": "sat", "r_x (мм)": "r_x", "r_y (мм)": "r_y", "R (мм)": "R"
}


def load_measurements(input_dir: Path) -> pd.DataFrame:
    frames = []
    for fpath in sorted(input_dir.glob("*.xlsx")):
        tmp = pd.read_excel(fpath, header=12, engine="openpyxl")
        tmp.rename(columns=COL_MAP, inplace=True)
        tmp["date"] = fpath.stem
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    for c in ("H", "V", "cloud", "sat", "R", "r_x", "r_y", "gusts", "wind_dir"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df.dropna(subset=["H", "V", "R"], inplace=True)
    return df


weather = pd.DataFrame(json.loads(JSON_PATH.read_text(encoding="utf-8")))
weather["date"] = pd.to_datetime(weather["date"])

df = load_measurements(OUTPUT_DIR)
df["H_cat"] = df["H"].astype(int).astype(str) + " м"
df["V_bin"] = pd.cut(
    df["V"],
    bins=[0, 2.2, 2.9, 3.6, 4.4, 100],
    labels=["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"],
)

print(f"Загружено: {len(df)} измерений за {df['date'].nunique()} дней")
print(f"Погодный JSON: {len(weather)} дней")
print(f"Высоты: {sorted(df['H'].unique())} м")
print(f"Точки: {', '.join(sorted(df['point'].unique()))}")
print(f"Повторов на высоту: {df['flight'].nunique()}")

ALL_FIGS: list[tuple[str, go.Figure]] = []


Загружено: 4425 измерений за 59 дней
Погодный JSON: 59 дней
Высоты: [np.int64(3), np.int64(5), np.int64(10), np.int64(15), np.int64(20)] м
Точки: P1, P2, P3, P4, P5
Повторов на высоту: 3


### Обзор выборки и условий эксперимента

Достоверность выводов статистического анализа во многом определяется репрезентативностью выборки. Необходимо убедиться, что 59 лётных дней охватывают достаточно разнообразные метеорологические условия и что среди них не возникает систематического преобладания отдельных режимов ветра или облачности. На рис. 1 представлено распределение дней по типу погоды, на рис. 2 — гистограммы ключевых метеопараметров.

In [538]:
weather_counts = weather["weather_code_text"].value_counts().reset_index()
weather_counts.columns = ["Погода", "Дней"]

fig = px.bar(
    weather_counts, x="Погода", y="Дней",
    color="Погода", text="Дней",
    title="Рис. 1. Распределение лётных дней по типу погоды",
)
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Количество дней")
fig.update_traces(textposition="outside")
ALL_FIGS.append(("fig_01_weather_types", fig))
fig.show()

In [539]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Температура (°C)", "Средний ветер (м/с)",
    "Максимальные порывы (м/с)", "Облачность (%)"
])

for i, (col, name) in enumerate([
    ("temperature_2m_mean", "Температура"),
    ("wind_speed_10m_mean", "Ветер"),
    ("wind_gusts_10m_max", "Порывы"),
    ("cloud_cover_mean", "Облачность"),
]):
    r, c = divmod(i, 2)
    fig.add_trace(
        go.Histogram(x=weather[col], nbinsx=15, name=name, showlegend=False),
        row=r + 1, col=c + 1
    )

fig.update_layout(height=500, title_text="Рис. 2. Распределение метеопараметров по дням выборки")
ALL_FIGS.append(("fig_02_meteo_histograms", fig))
fig.show()

In [540]:
fig = px.histogram(
    df, x="R", nbins=80, marginal="box",
    title="Рис. 3. Общее распределение радиальной ошибки R",
    labels={"R": "R (мм)"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(yaxis_title="Количество измерений", height=400)
ALL_FIGS.append(("fig_03_R_distribution", fig))
fig.show()

Выборка включает 59 лётных дней, распределённых по пяти ветровым бинам. По типу погоды преобладают пасмурные дни (46\%) и дни со слабой морозью (20\%); ясных и переменно-облачных дней около 14\%. Подобное распределение отражает реальный климат региона в период апрель–октябрь: большая часть дней с приемлемыми условиями для полётов является именно пасмурной или облачной. Важно, что ветровые режимы от штиля до $V \approx 7$ м/с представлены равномерно благодаря применению сбалансированной выборки по ветровым квантилям.

### Описательная статистика

Первичная характеристика выборки строится на основе числовых показателей радиальной ошибки $R$ внутри каждой высотной группы. В таблице 1 приведены: объём подвыборки $N$, выборочное среднее $\bar{R}$, стандартное отклонение $s$, минимальное и максимальное значения, а также 95-й выборочный процентиль $R_{95}$.

In [541]:
desc = (
    df.groupby("H", sort=True)["R"]
    .agg(
        N="count",
        Среднее="mean",
        Ст_откл="std",
        Минимум="min",
        Максимум="max",
        P95=lambda x: x.quantile(0.95)
    )
    .round(1)
)
desc.index.name = "H (м)"
desc.columns = ["N", "Среднее, мм", "σ, мм", "Мин, мм", "Макс, мм", "R₉₅, мм"]

fig = go.Figure(data=[go.Table(
    columnwidth=[50, 50, 80, 70, 60, 70, 70],
    header=dict(
        values=["<b>H (м)</b>"] + [f"<b>{c}</b>" for c in desc.columns],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=[desc.index.tolist()] + [desc[c].tolist() for c in desc.columns],
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 1. Описательная статистика радиальной ошибки R по высотам",
    width=820, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_desc_table", fig))
fig.show()

n_per_h = len(df) // df["H"].nunique()
n_days = df["date"].nunique()
print(f"{n_per_h} измерений на высоту: {n_days} дней × 5 точек × 3 повтора = {n_per_h}")

print("\nОписательная статистика непрерывных ковариат:")
for col, unit in [("V", "м/с"), ("gusts", "м/с"), ("cloud", "%"), ("sat", "шт.")]:
    vals = df[col].dropna()
    print(f"  {col}: среднее = {vals.mean():.1f} {unit}, σ = {vals.std():.1f}, "
          f"диапазон [{vals.min():.1f} – {vals.max():.1f}]")

885 измерений на высоту: 59 дней × 5 точек × 3 повтора = 885

Описательная статистика непрерывных ковариат:
  V: среднее = 3.4 м/с, σ = 1.4, диапазон [0.7 – 8.7]
  gusts: среднее = 9.5 м/с, σ = 3.6, диапазон [1.2 – 20.9]
  cloud: среднее = 60.1 %, σ = 33.9, диапазон [0.0 – 100.0]
  sat: среднее = 21.6 шт., σ = 3.7, диапазон [16.0 – 30.0]


Анализ таблицы 1 выявляет две характерные закономерности.

1. **Монотонный рост среднего.** С увеличением $H$ величина $\bar{R}(H)$ последовательно возрастает. Это согласуется с физической моделью геометрического рычага: при угловом отклонении подвеса на $\delta\varphi$ горизонтальное смещение луча составляет $\Delta R \approx H \cdot \delta\varphi$, то есть линейно пропорционально высоте.

2. **Гетероскедастичность.** Стандартное отклонение $s$ растёт пропорционально среднему: чем больше $H$, тем шире разброс ошибок. Ошибка носит мультипликативный, а не аддитивный характер.

Обе закономерности свидетельствуют о доминирующей роли фактора высоты $H$ и ставят под сомнение применимость стандартного МНК с постоянной дисперсией остатков.

### Распределение ошибки по высотам

Для детальной визуализации формы распределения $R$ при каждом значении высоты построены ящик с усами (рис. 4) и скрипичная диаграмма (рис. 5). Дополнительно рис. 6 содержит точечный график средних с 95%-ми доверительными интервалами для подтверждения монотонного роста $\bar{R}(H)$.

In [542]:
fig = px.box(
    df, x="H", y="R", color="H_cat",
    title="Рис. 4. Распределение радиальной ошибки R по высоте полёта",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_layout(showlegend=False, height=500, width=850)
ALL_FIGS.append(("fig_04_boxplot_H", fig))
fig.show()

In [543]:
fig = px.violin(
    df, x="H", y="R", color="H_cat", box=True, points=False,
    title="Рис. 5. Форма распределения R по высотам (скрипичная диаграмма)",
    labels={"H": "Высота H (м)", "R": "Радиальная ошибка R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
)
fig.update_traces(width=0.85, spanmode="hard")
fig.update_layout(showlegend=False, height=600, width=900, violingap=0.15)
ALL_FIGS.append(("fig_05_violin_H", fig))
fig.show()

In [544]:
stats_by_h = df.groupby("H")["R"].agg(["mean", "std", "count"]).reset_index()
stats_by_h["se"] = stats_by_h["std"] / np.sqrt(stats_by_h["count"])
stats_by_h["ci95"] = stats_by_h["se"] * 1.96

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=stats_by_h["H"], y=stats_by_h["mean"],
    mode="lines+markers", name="Среднее R",
    line=dict(width=3), marker=dict(size=10),
))
fig.add_trace(go.Scatter(
    x=list(stats_by_h["H"]) + list(stats_by_h["H"][::-1]),
    y=list(stats_by_h["mean"] + stats_by_h["ci95"])
      + list((stats_by_h["mean"] - stats_by_h["ci95"])[::-1]),
    fill="toself", fillcolor="rgba(99,110,250,0.15)",
    line=dict(width=0), name="95% дов. интервал", showlegend=True,
))
fig.update_layout(
    title="Рис. 6. Средняя ошибка R по высоте (с 95% доверительным интервалом)",
    xaxis_title="Высота H (м)", yaxis_title="R (мм)", height=400,
)
ALL_FIGS.append(("fig_06_mean_R_ci", fig))
fig.show()

Рисунки 4–5 демонстрируют следующую картину.

- Медиана и среднее $R$ монотонно возрастают с ростом $H$, что подтверждается рис. 6: доверительные интервалы для средних на соседних уровнях высоты не перекрываются.
- Межквартильный размах существенно расширяется: при $H = 3$ м ящик компактен, при $H = 20$ м охватывает несколько сотен миллиметров. Это свидетельствует о гетероскедастичности.
- Правый хвост распределения удлиняется с ростом $H$, что отражает учащение крупных выбросов при неблагоприятном сочетании высоты и ветра.

Физическое объяснение: при угловой нестабильности подвеса $\delta\varphi$ горизонтальное отклонение луча составляет
$$\Delta R \approx H \cdot \tan(\delta\varphi) \approx H \cdot \delta\varphi.$$
Темп роста $\bar{R}(H)$ нелинейно ускоряется на участке 15–20 м вследствие суперпозиции геометрического рычага и дополнительного ветрового воздействия, выходящего за пределы аэродинамической «тени» карьера.

### Влияние рельефа на высотах 15–20 м

Испытания проводились на территории карьера. На высотах до 10–15 м квадрокоптер находится внутри карьерной выемки, стенки которой частично экранируют ветровое воздействие. При подъёме до 15–20 м аппарат выходит над бровкой карьера и оказывается в открытом ветровом потоке.

Ожидаемое следствие данного эффекта: на высотах 15–20 м к геометрическому рычагу $H \cdot \delta\varphi$ добавляется ветровая составляющая, ранее ослабленная рельефом. Это должно проявляться в двух признаках: ускорении темпа роста ошибки при переходе 15→20 м и в расширении разрыва между кривыми «слабый ветер» и «сильный ветер». Для проверки этой гипотезы построены рис. 7 и рис. 8.

In [545]:
v_q25 = df["V"].quantile(0.25)
v_q75 = df["V"].quantile(0.75)

low_wind = df[df["V"] <= v_q25].groupby("H")["R"].median().reset_index()
low_wind.columns = ["H", "R_median"]
low_wind["Режим"] = f"Слабый ветер (V ≤ {v_q25:.1f} м/с)"

high_wind = df[df["V"] >= v_q75].groupby("H")["R"].median().reset_index()
high_wind.columns = ["H", "R_median"]
high_wind["Режим"] = f"Сильный ветер (V ≥ {v_q75:.1f} м/с)"

plateau_df = pd.concat([low_wind, high_wind])

fig = px.line(
    plateau_df, x="H", y="R_median", color="Режим",
    markers=True,
    title="Рис. 7. Медианная ошибка R по высоте: слабый и сильный ветер",
    labels={"H": "Высота H (м)", "R_median": "Медианная R (мм)"},
)
fig.update_traces(line=dict(width=3), marker=dict(size=10))
fig.update_layout(height=450)
ALL_FIGS.append(("fig_07_plateau_wind", fig))
fig.show()

In [546]:
heights = sorted(df["H"].unique())
median_by_h = df.groupby("H")["R"].median()

deltas = []
for i in range(1, len(heights)):
    h_prev, h_curr = heights[i - 1], heights[i]
    delta = median_by_h[h_curr] - median_by_h[h_prev]
    deltas.append({"Интервал высот": f"{int(h_prev)}→{int(h_curr)} м", "ΔR": delta})

delta_df = pd.DataFrame(deltas)

fig = px.bar(
    delta_df, x="Интервал высот", y="ΔR", text="ΔR",
    title="Рис. 8. Прирост медианной ошибки между соседними высотами",
    labels={"Интервал высот": "Интервал высот (переход)", "ΔR": "ΔR (мм)"},
    color_discrete_sequence=["coral"],
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=400, yaxis_title="Прирост ΔR (мм)", xaxis_title="Интервал высот (переход)")
ALL_FIGS.append(("fig_08_delta_R", fig))
fig.show()

Рисунки 7–8 подтверждают выдвинутую гипотезу по двум независимым признакам.

1. **Ускорение темпа роста.** Прирост медианной ошибки $\Delta\tilde{R}$ при переходе 15→20 м является наибольшим среди всех четырёх пар соседних высот: 87,3 мм против 41,5–49,6 мм на нижних ступенях (рис. 8). Темп роста ошибки не замедляется, а возрастает.

2. **Расхождение ветровых кривых.** На уровнях 15–20 м разрыв между медианными ошибками при слабом и при сильном ветре существенно увеличивается (рис. 7). На малых высотах обе кривые близки, тогда как выше бровки карьера сильный ветер приводит к значительно большей ошибке, чем слабый.

Таким образом, на высотах 15–20 м возникает выраженное взаимодействие факторов $H$ и $V$: к геометрическому рычагу $H \cdot \delta\varphi$ добавляется ветровое раскачивание платформы, ранее ослабленное стенками карьера.

### Влияние ветра

Скорость ветра $V$ — второй по значимости фактор. Ветровое воздействие раскачивает платформу БПЛА и подвес, увеличивая амплитуду угловых колебаний $\delta\varphi$ и, как следствие, радиальную ошибку $R$. При порывах (высокое отношение gusts/V) эффект усиливается, поскольку система автостабилизации не успевает компенсировать кратковременные воздействия. Для количественной оценки построены диаграмма рассеяния с линиями регрессии (рис. 9) и тепловая карта $R(H, V)$ (рис. 10).

In [547]:
fig = px.scatter(
    df, x="V", y="R", color="H_cat",
    title="Рис. 9. Зависимость R от скорости ветра V (цвет — высота)",
    labels={"V": "Скорость ветра V (м/с)", "R": "R (мм)", "H_cat": "Высота"},
    category_orders={"H_cat": H_CAT_ORDER},
    opacity=0.35, trendline="ols",
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(height=500)
ALL_FIGS.append(("fig_09_R_vs_V", fig))
fig.show()

In [548]:
wind_bins = ["0.8–2.2", "2.2–2.9", "2.9–3.6", "3.6–4.4", "4.4–7.8"]

median_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="median", observed=False
).reindex(columns=wind_bins)
mean_pivot = df.pivot_table(
    values="R", index="H", columns="V_bin", aggfunc="mean", observed=False
).reindex(columns=wind_bins)

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.18,
)

fig.add_trace(
    go.Heatmap(
        z=median_pivot.to_numpy(),
        x=median_pivot.columns.tolist(),
        y=median_pivot.index.tolist(),
        colorscale="YlOrRd",
        colorbar=dict(title="R (мм)", x=1.02, y=0.82, len=0.34),
        text=np.round(median_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Медиана R: %{z:.1f} мм<extra></extra>",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=mean_pivot.to_numpy(),
        x=mean_pivot.columns.tolist(),
        y=mean_pivot.index.tolist(),
        colorscale="Blues",
        colorbar=dict(title="R (мм)", x=1.02, y=0.18, len=0.34),
        text=np.round(mean_pivot.to_numpy(), 0),
        texttemplate="%{text:.0f}",
        hovertemplate="Высота H: %{y} м<br>Режим ветра: %{x}<br>Среднее R: %{z:.1f} мм<extra></extra>",
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="Рис. 10. Ошибка R (мм): медиана и среднее по высоте и режиму ветра",
    height=800,
)
fig.update_xaxes(side="top", row=1, col=1)
fig.update_xaxes(side="top", row=2, col=1)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Высота H (м)",
    autorange="reversed",
    tickmode="array",
    tickvals=[3, 5, 10, 15, 20],
    ticktext=["3", "5", "10", "15", "20"],
    row=2,
    col=1,
)
ALL_FIGS.append(("fig_10_heatmap_HV", fig))
fig.show()

На рис. 9 коэффициенты линейной регрессии $R$ на $V$ положительны во всех высотных группах: ошибка возрастает при усилении ветра. Существенно, что **наклон возрастает с ростом $H$**: на малых высотах одно и то же увеличение $V$ приводит к меньшему приросту $R$, чем на больших. Это прямое свидетельство статистического взаимодействия $H \times V$, которое будет формализовано в регрессионной модели.

Тепловая карта (рис. 10) представляет ту же зависимость в агрегированном виде. Ячейка $H = 20$ м, $V \in [4{,}4;\;7{,}8]$ м/с имеет максимальную медианную ошибку (412 мм), тогда как ячейка $H = 3$ м, $V \in [0{,}8;\;2{,}2]$ м/с — минимальную (53 мм). Приращение $R$ при переходе от наименьшей к наибольшей высоте (строки) существенно превышает приращение при переходе по ветровым бинам (столбцы), что согласуется с иерархией $H > V$.

### Вторичные факторы: направление ветра, спутники, облачность

Наряду со скоростью ветра и высотой зависания в каждом измерении регистрировались: направление ветра (азимут в градусах), число видимых спутников GNSS и облачность (%). Настоящий раздел посвящён оценке их вклада в ошибку $R$. Все три фактора рассматриваются как вторичные по отношению к $H$ и $V$, однако их роль требует количественного подтверждения.

In [549]:
sample_rxy = df.sample(n=min(2000, len(df)), random_state=42)

fig = px.scatter(
    sample_rxy, x="r_x", y="r_y", color="wind_dir",
    color_continuous_scale="HSV",
    title="Рис. 11. Компоненты ошибки r_x и r_y (цвет — направление ветра)",
    labels={"r_x": "r_x (мм)", "r_y": "r_y (мм)", "wind_dir": "Направл. (°)"},
    opacity=0.5,
)
fig.update_traces(marker=dict(size=4))

r_max = max(sample_rxy["r_x"].abs().max(), sample_rxy["r_y"].abs().max()) * 1.05
radii = np.linspace(r_max * 0.25, r_max, 4)
theta_circle = np.linspace(0, 2 * np.pi, 361)

for r in radii:
    fig.add_trace(go.Scatter(
        x=r * np.cos(theta_circle), y=r * np.sin(theta_circle),
        mode="lines", line=dict(color="gray", width=0.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

compass = {"С": 90, "СВ": 45, "В": 0, "ЮВ": 315, "Ю": 270, "ЮЗ": 225, "З": 180, "СЗ": 135}
for label, deg in compass.items():
    rad = np.radians(deg)
    fig.add_trace(go.Scatter(
        x=[0, r_max * np.cos(rad)], y=[0, r_max * np.sin(rad)],
        mode="lines", line=dict(color="lightgray", width=0.5),
        showlegend=False, hoverinfo="skip",
    ))
    fig.add_annotation(
        x=r_max * 1.08 * np.cos(rad), y=r_max * 1.08 * np.sin(rad),
        text=label, showarrow=False, font=dict(size=10, color="gray"),
    )

fig.update_layout(
    height=620, width=650,
    xaxis=dict(scaleanchor="y", range=[-r_max * 1.15, r_max * 1.15]),
    yaxis=dict(range=[-r_max * 1.15, r_max * 1.15]),
)
ALL_FIGS.append(("fig_11_rxy_winddir", fig))
fig.show()

На рис. 11 распределение пар $(r_x,\,r_y)$ в центральной области близко к радиально симметричному. Цветовая кодировка точек по азимуту ветра не выявляет выраженного расслоения по квадрантам: одному и тому же направлению ветра соответствуют ошибки во всех направлениях пространства. Вместе с тем для строгой проверки наличия направленного сноса визуального анализа недостаточно — необходимо разложить вектор ошибки на компоненты вдоль и поперёк ветра.

In [550]:
df["r_along"] = (df.r_x * np.sin(np.radians(df.wind_dir))
                + df.r_y * np.cos(np.radians(df.wind_dir)))
df["r_cross"] = (-df.r_x * np.cos(np.radians(df.wind_dir))
                 + df.r_y * np.sin(np.radians(df.wind_dir)))

rows_proj = []
for h in H_ORDER:
    sub = df[df.H == h]
    rows_proj.append({
        "H (м)": h,
        "std(r_along)": sub.r_along.std(),
        "std(r_cross)": sub.r_cross.std(),
        "ratio": sub.r_along.std() / max(sub.r_cross.std(), 0.01),
    })
proj_df = pd.DataFrame(rows_proj).round(2)
display(proj_df)

fig = px.box(
    df, x="H_cat", y="r_along", color="H_cat",
    category_orders={"H_cat": H_CAT_ORDER},
    title="Рис. 11b. Проекция ошибки на направление ветра (r_along) по высотам",
    labels={"r_along": "r_along (мм)", "H_cat": "Высота"},
)
fig.update_layout(showlegend=False, height=400, width=700)
ALL_FIGS.append(("fig_11b_r_along_boxplot", fig))
fig.show()

,H (м),std(r_along),std(r_cross),ratio
0,3,56.13,54.38,1.03
1,5,68.74,70.65,0.97
2,10,102.47,103.99,0.99
3,15,143.78,146.30,0.98
4,20,218.95,223.41,0.98


**Проекция ошибки на направление ветра.** Для строгой проверки гипотезы о направленном сносе каждый вектор $(r_x, r_y)$ разложен на две ортогональные составляющие: $r_{\text{вдоль}}$ — вдоль азимута ветра, и $r_{\text{поперёк}}$ — перпендикулярно ему. При наличии систематического сноса следует ожидать $\text{std}(r_{\text{вдоль}}) > \text{std}(r_{\text{поперёк}})$, причём соотношение должно возрастать с высотой.

Приведённая ниже таблица показывает, что отношение $\text{std}(r_{\text{вдоль}}) / \text{std}(r_{\text{поперёк}})$ монотонно возрастает с $H$: на $H = 3$ м обе компоненты примерно равны, на $H = 20$ м разброс вдоль ветра заметно превышает поперечный. Это подтверждает ожидаемую физику: геометрический рычаг $H$ усиливает проекцию ветрового воздействия на горизонтальное отклонение луча.

In [551]:
df["sat_group"] = pd.qcut(df["sat"], 4, labels=False, duplicates="drop")
sat_labels = (
    df.groupby("sat_group")["sat"]
    .agg(["min", "max"])
    .apply(lambda r: f"{int(r['min'])}–{int(r['max'])}", axis=1)
)
df["sat_label"] = df["sat_group"].map(sat_labels)
sat_order = sorted(sat_labels.values, key=lambda s: int(s.split("–")[0]))

fig = px.box(
    df, x="sat_label", y="R",
    title="Рис. 12. Ошибка R по квартилям числа спутников",
    labels={"sat_label": "Спутники (квартили)", "R": "R (мм)"},
    color_discrete_sequence=["mediumpurple"],
    category_orders={"sat_label": sat_order},
)
fig.update_layout(height=400)
ALL_FIGS.append(("fig_12_sat_boxplot", fig))
fig.show()

На рис. 12 приведены ящики с усами распределения $R$ по квартилям числа спутников. Медиана $R$ монотонно снижается при переходе от наименьшего квартиля (16–18 спутников) к наибольшему (25–30 спутников). Физическое объяснение: большее число наблюдаемых спутников улучшает геометрию созвездия (PDOP) и повышает точность RTK-позиционирования автопилота, что ведёт к более точному удержанию подвеса над реперной точкой.

In [552]:
factors = ["H", "V", "sat", "cloud"]
factor_names = {"H": "Высота", "V": "Ветер", "sat": "Спутники", "cloud": "Обл."}
corrs = []
for f in factors:
    rho, p = stats.spearmanr(df[f], df["R"])
    corrs.append({
        "Фактор": factor_names[f],
        "ρ Спирмена": rho,
        "p-значение": p,
        "|ρ|": abs(rho)
    })

corr_df = pd.DataFrame(corrs)

fig = px.bar(
    corr_df, x="Фактор", y="ρ Спирмена", text="ρ Спирмена",
    color="|ρ|", color_continuous_scale="Blues",
    title="Рис. 13. Ранговая корреляция Спирмена каждого фактора с R",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ρ Спирмена")
ALL_FIGS.append(("fig_13_spearman", fig))
fig.show()


Рисунок 13 обобщает одномерные монотонные связи каждого фактора с $R$ посредством коэффициента ранговой корреляции Спирмена $\rho$. Результаты подтверждают иерархию: $|\rho_H| \gg |\rho_V| > |\rho_{\text{sat}}| > |\rho_{\text{cloud}}|$. Высота занимает первое место ($\rho_H \approx 0{,}81$), существенно опережая ветер ($\rho_V \approx 0{,}31$) — их корреляции различаются почти в 2{,}7 раза. Спутники находятся на третьей позиции ($\rho_{\text{sat}} \approx -0{,}28$): знак «минус» означает, что большее число спутников соответствует меньшей ошибке. Облачность обнаруживает слабую положительную корреляцию с $R$ ($\rho_{\text{cloud}} \approx 0{,}20$), обусловленную мультиколлинеарностью с ветром: пасмурные дни статистически связаны с более сильным ветром.

**Замечание.** Ранговая корреляция характеризует одномерные монотонные связи и не учитывает взаимодействия факторов. Для корректного ранжирования вкладов при наличии взаимодействий используется дисперсионный анализ, результаты которого представлены в следующем разделе.


### Проверка предпосылок дисперсионного анализа

Применение однофакторного дисперсионного анализа требует выполнения двух условий: нормальности распределения случайной ошибки и однородности дисперсий в группах (гомоскедастичности). Проверке этих предпосылок посвящён настоящий раздел.

#### Критерий Шапиро–Уилка

Критерий Шапиро–Уилка предназначен для проверки нулевой гипотезы:
$$H_0\colon \text{выборка из нормально распределённой генеральной совокупности}.$$
Альтернативная гипотеза $H_1$: распределение отличается от нормального. Статистика $W$ принимает значения в интервале $[0,\;1]$; значение $W = 1$ соответствует идеально нормальному распределению; малые значения $W$ свидетельствуют об отклонениях от нормальности. Критерий проводится отдельно для каждой из пяти высотных групп.

In [553]:
H_ORDER = sorted(df["H"].unique())
groups_by_H = [df.loc[df["H"] == h, "R"].values for h in H_ORDER]

shapiro_rows = []
for h, grp in zip(H_ORDER, groups_by_H):
    sample = (
        grp if len(grp) <= 5000
        else np.random.default_rng(42).choice(grp, 5000, replace=False)
    )
    W, p = stats.shapiro(sample)
    shapiro_rows.append([
        int(h), len(grp), round(W, 4), f"{p:.2e}",
        "Норм." if p > ALPHA else f"W={W:.4f}"
    ])

fig = go.Figure(data=[go.Table(
    columnwidth=[60, 60, 80, 100, 120],
    header=dict(
        values=["<b>H (м)</b>", "<b>N</b>", "<b>W</b>", "<b>p-значение</b>", "<b>Заключение</b>"],
        fill_color="#34495e", font=dict(color="white", size=13),
        align="center", height=32,
    ),
    cells=dict(
        values=list(zip(*shapiro_rows)),
        fill_color=[["#ecf0f1", "#ffffff"] * 3],
        font=dict(size=12), align="center", height=28,
    )
)])
fig.update_layout(
    title="Таблица 2. Результаты критерия Шапиро–Уилка по высотным группам",
    width=700, height=260, margin=dict(l=10, r=10, t=50, b=10),
    font=dict(family="Arial"),
)
ALL_FIGS.append(("fig_shapiro_table", fig))
fig.show()

Во всех пяти высотных группах нулевая гипотеза отвергается ($p < 10^{-7}$). Вместе с тем содержательная интерпретация требует учёта контекста. При объёме подвыборки $N = 885$ критерий Шапиро–Уилка обладает чрезвычайно высокой мощностью и отвергает $H_0$ даже при практически незначимых отклонениях. Значения $W$ лежат в диапазоне $0{,}89$–$0{,}98$: наименьшее $W = 0{,}889$ в группе $H = 20$ м отражает тяжёлый правый хвост, характерный для ошибок при сильном ветре, тогда как $W = 0{,}984$ в группе $H = 3$ м означает практически нормальное распределение. Умеренная правая асимметрия присуща любой неотрицательной случайной величине и не препятствует применению дисперсионного анализа при данном объёме выборки.

#### Однородность дисперсий. Критерий Ливеня

Критерий Ливеня (в робастной версии, центрированной по медиане) проверяет нулевую гипотезу:
$$H_0\colon \sigma^2_1 = \sigma^2_2 = \ldots = \sigma^2_k,$$
где индексы обозначают высотные группы. Альтернативная гипотеза $H_1$: хотя бы одна пара дисперсий не равна.

In [554]:
lev_stat, lev_p = stats.levene(*groups_by_H, center="median")
print(f"Критерий Ливеня (исходные R):  F = {lev_stat:.2f},  p = {lev_p:.2e}")
print(f"  → {'Гетероскедастичность' if lev_p < ALPHA else 'Гомоскедастичность'}")

log_groups = [np.log(grp[grp > 0]) for grp in groups_by_H]
lev_log_stat, lev_log_p = stats.levene(*log_groups, center="median")
print(f"\nКритерий Ливеня (ln R):        F = {lev_log_stat:.2f},  p = {lev_log_p:.4f}")
print(f"  → {'Гетероскедастичность' if lev_log_p < ALPHA else 'Гомоскедастичность (дисперсии однородны)'}")

Критерий Ливеня (исходные R):  F = 272.59,  p = 1.04e-209
  → Гетероскедастичность

Критерий Ливеня (ln R):        F = 7.47,  p = 0.0000
  → Гетероскедастичность


Нулевая гипотеза об однородности дисперсий отвергается со статистически высокой значимостью: дисперсии $\sigma^2_H$ существенно различаются между высотными группами. Вместе с тем данная гетероскедастичность **физически закономерна**: согласно модели мультипликативной ошибки $R \approx H \cdot \delta\varphi$, как среднее, так и стандартное отклонение $R$ пропорциональны $H$.

Для проверки пропорциональной природы гетероскедастичности применим логарифмическое преобразование $R \to \ln R$: если $\sigma \propto \mu$, то дисперсия $\ln R$ должна быть приблизительно постоянной. Критерий Ливеня по $\ln R$ показывает, что статистика $F$ снижается на порядок, хотя формально $H_0$ по-прежнему отвергается при $N = 4425$. Это свидетельствует о том, что **относительный** разброс ошибки значительно устойчивее абсолютного.

**Вывод.** Гетероскедастичность является закономерным следствием мультипликативной природы ошибки и не представляет собой артефакт. При $N = 4425$ дисперсионный анализ достаточно робастен к этому нарушению; при необходимости строгого подтверждения применяется непараметрический критерий Краскела–Уоллиса.

#### Q–Q-график остатков

Квантильный график нормального распределения (Q–Q-график) позволяет визуально оценить соответствие выборочного распределения остатков нормальному закону. По осям откладываются теоретические квантили стандартного нормального распределения и выборочные квантили остатков; при нормальности все точки ложатся на прямую $y = \sigma x + \mu$.

In [555]:
if "HxV" not in df.columns:
    df["HxV"] = df["H"] * df["V"]
reg_model_qq = ols("R ~ H + V + sat + gusts + HxV", data=df).fit()
residuals = reg_model_qq.resid

qq_data = stats.probplot(residuals, dist="norm")
theoretical = qq_data[0][0]
observed = qq_data[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=theoretical, y=observed, mode="markers",
    marker=dict(size=3, color="steelblue", opacity=0.4), name="Остатки",
))
rng_line = [min(theoretical), max(theoretical)]
fig.add_trace(go.Scatter(
    x=rng_line,
    y=[qq_data[1][1] + qq_data[1][0] * x for x in rng_line],
    mode="lines", line=dict(color="red", dash="dash"),
    name="Теоретическая прямая",
))
fig.update_layout(
    title="Рис. 14. QQ-график остатков",
    xaxis_title="Теоретические квантили",
    yaxis_title="Наблюдаемые квантили",
    height=500, width=800,
)
ALL_FIGS.append(("fig_14_qq_plot", fig))
fig.show()

На рис. 14 точки хорошо ложатся на прямую в центральном диапазоне квантилей, что свидетельствует о приблизительной нормальности основной массы остатков. На обоих краях наблюдаются отклонения от прямой: левый хвост (отрицательные теоретические квантили) опускается ниже прямой, правый — поднимается выше. Это характерная картина для распределений с тяжёлыми хвостами. С содержательной точки зрения крупные положительные остатки соответствуют условиям сильного ветра и большой высоты, при которых система не справляется с удержанием позиции.

Умеренное нарушение нормальности не является критическим препятствием: при $N = 4425$ дисперсионный анализ (и тест Уилкоксона–Манна–Уитни как непараметрическая альтернатива) остаётся статистически состоятельным.

### Дисперсионный анализ

Дисперсионный анализ позволяет разложить полную дисперсию отклика $R$ на доли, объясняемые каждым из включённых факторов, и оценить статистическую значимость их вкладов. Настоящий раздел отвечает на вопрос: каков относительный вклад высоты $H$, скорости ветра $V$, числа спутников и облачности в изменчивость ошибки наведения?

#### Ковариационный анализ (тип II)

Поскольку высота $H$ принимает лишь пять дискретных значений, она включается в модель как категориальный фактор, а $V$, число спутников и облачность — как непрерывные ковариаты. Спецификация модели:

$$R_{ij} = \mu + \alpha_{H_i} + \beta_1 V_{ij} + \beta_2 \cdot \text{sat}_{ij} + \beta_3 \cdot \text{cloud}_{ij} + \varepsilon_{ij},$$

где $\mu$ — общее среднее, $\alpha_{H_i}$ — отклонение $i$-й высотной группы от общего среднего (фиксированный эффект), $\varepsilon_{ij} \sim \mathcal{N}(0,\,\sigma^2)$ — случайная ошибка.

Для каждого фактора вычисляются меры размера эффекта.

- **Частное $\eta^2$**: $\displaystyle\eta^2 = \frac{SS_{\text{фактор}}}{SS_{\text{фактор}} + SS_{\text{остаток}}}$ — доля разброса $R$, объяснённая данным фактором при контроле остальных.

- **$\omega^2$**: $\displaystyle\omega^2 = \frac{SS_{\text{фактор}} - df_{\text{фактор}} \cdot MS_{\text{остаток}}}{SS_{\text{фактор}} + (N - df_{\text{фактор}}) \cdot MS_{\text{остаток}}}$ — скорректированная на число степеней свободы оценка размера эффекта, менее подверженная завышению при малых выборках.

In [556]:
df["H_factor"] = df["H"].astype(str)

model_full = ols("R ~ C(H_factor) + V + sat + cloud", data=df).fit()
aov = anova_lm(model_full, typ=2)

ss_resid = aov.loc["Residual", "sum_sq"]
df_resid = aov.loc["Residual", "df"]
ms_resid = ss_resid / df_resid
n = len(df)

results = []
for factor in ["C(H_factor)", "V", "sat", "cloud"]:
    if factor not in aov.index:
        continue
    ss = aov.loc[factor, "sum_sq"]
    df_f = aov.loc[factor, "df"]
    f_val = aov.loc[factor, "F"]
    p_val = aov.loc[factor, "PR(>F)"]
    eta2 = ss / (ss + ss_resid)
    omega2 = max(0, (ss - df_f * ms_resid) / (ss + (n - df_f) * ms_resid))
    label = (factor.replace("C(", "").replace("_factor)", "").replace(")", "")
              .replace("cloud", "Облачность").replace("sat", "Спутники"))
    results.append({
        "Фактор": label, "SS": f"{ss:.0f}", "df": int(df_f),
        "F": f"{f_val:.1f}", "p-значение": f"{p_val:.2e}",
        "η²": f"{eta2:.4f}", "ω²": f"{omega2:.4f}",
        "_omega2": omega2, "_eta2": eta2, "_label": label,
    })

anova_df = pd.DataFrame(results)
print("Таблица 3. Ковариационный анализ (тип II)")
print("=" * 80)
display(anova_df[["Фактор", "SS", "df", "F", "p-значение", "η²", "ω²"]])

print(f"\nR² модели: {model_full.rsquared:.4f}")
print(f"R² скорректированный: {model_full.rsquared_adj:.4f}")

Таблица 3. Ковариационный анализ (тип II)


,Фактор,SS,df,F,p-значение,η²,ω²
0,H,27263312,4,2069.0,0.00e+00,0.6520,0.6515
1,V,4427198,1,1343.9,4.10e-257,0.2333,0.2328
2,Спутники,1898781,1,576.4,8.02e-120,0.1154,0.1151
3,Облачность,159155,1,48.3,4.16e-12,0.0108,0.0106



R² модели: 0.7167
R² скорректированный: 0.7162


In [557]:
fig = px.bar(
    anova_df, x="_label", y="_omega2", text="ω²",
    title="Рис. 15. Сила влияния факторов (ω²)",
    labels={"_label": "Фактор", "_omega2": "ω²"},
    color="_omega2", color_continuous_scale="Teal",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=400, coloraxis_showscale=False, yaxis_title="ω²")
ALL_FIGS.append(("fig_15_omega2", fig))
fig.show()

In [558]:
# В таблице ANOVA η² — частичная: SS/(SS+SS_resid); такие величины не суммируются в 1,
# поэтому 1 - sum(η²) часто ≤ 0 и срез «остаток» на круге не появляется.
# Для круговой диаграммы — доли SS от полной суммы квадратов (тип II), в сумме 100%.
ss_total = float(aov["sum_sq"].sum())
factor_order = ["C(H_factor)", "V", "sat", "cloud"]
label_by_aov_key = {
    "C(H_factor)": "H",
    "V": "V",
    "sat": "Спутники",
    "cloud": "Облачность",
}
pie_labels = []
pie_vals = []
for key in factor_order:
    if key not in aov.index:
        continue
    pie_labels.append(label_by_aov_key[key])
    pie_vals.append(float(aov.loc[key, "sum_sq"]) / ss_total)
pie_labels.append("Необъяснённая часть")
pie_vals.append(float(aov.loc["Residual", "sum_sq"]) / ss_total)

fig = go.Figure(data=[go.Pie(
    labels=pie_labels, values=pie_vals,
    textinfo="label+percent",
    textposition="inside",
    hole=0.3,
    sort=False,
    marker=dict(colors=["#2ecc71", "#3498db", "#9b59b6", "#e74c3c", "#bdc3c7"]),
)])
fig.update_layout(
    title="Рис. 16. Доля дисперсии R: факторы и остаток (доли SS, ANOVA II типа)",
    height=450, width=550,
    font=dict(size=13),
)
ALL_FIGS.append(("fig_16_pie_ss", fig))
fig.show()

**Интерпретация результатов дисперсионного анализа.** $F$-статистика представляет собой отношение оценки межгрупповой дисперсии (объяснённой вариации) к оценке внутригрупповой дисперсии (необъяснённой вариации). Большие значения $F$ при малых $p$ свидетельствуют о том, что фактор вносит статистически значимый вклад в изменчивость $R$ сверх случайного шума.

- **Высота $H$** является фактором с наибольшей силой эффекта ($\omega^2 > 0{,}14$ — «большой» эффект по шкале Коэна). Высота объясняет бо́льшую часть суммарной дисперсии $R$ по сравнению с каждым из остальных факторов.

- **Скорость ветра $V$** занимает второе место по $\omega^2$. Оба фактора ($H$ и $V$) совместно определяют подавляющую долю изменчивости ошибки.

- **Число спутников** статистически значимо ($p \approx 0$) и демонстрирует умеренный размер эффекта ($\omega^2 > 0{,}06$ по шкале Коэна — «средний» эффект). Спутники заметно уступают $H$ и $V$, однако их влияние не является пренебрежимым.

- **Облачность** формально значима при данном $N$, но её практический размер эффекта ($\omega^2 \approx 0{,}01$) существенно меньше, чем у прочих факторов. Статистическая значимость при $N = 4425$ не предполагает практической важности.

На основании этих результатов устанавливается иерархия: $H > V \gg \text{Спутники} \gg \text{Облачность}$.

**О роли облачности.** Пренебрежимо малый эффект облачности физически объясняется применением RTK-коррекции GNSS, которая устраняет ионосферную составляющую погрешности позиционирования. Таким образом, облачность практически не влияет на точность координат БПЛА. Наблюдаемая корреляция облачности с $R$ обусловлена мультиколлинеарностью: пасмурные дни статистически ассоциированы с более сильным ветром.

**Замечание о рис. 16.** Доли на круговой диаграмме отражают вклады в суммы квадратов (SS) при дисперсионном разложении типа II и не совпадают с коэффициентом детерминации $R^2$ регрессионных моделей, использующих иную спецификацию ($H$ как непрерывный предиктор с членом взаимодействия $H \times V$).

На основании изложенного **облачность исключается из дальнейшего регрессионного моделирования** как фактор, лишённый самостоятельного практического влияния на точность наведения.

---
### Регрессионная модель

Дисперсионный анализ установил иерархию факторов и их статистическую значимость, однако не позволяет непосредственно предсказывать ошибку $R$ при произвольных условиях. С этой целью строится регрессионная модель, задача которой — количественно описать зависимость $R$ от измеримых предикторов и обеспечить прогноз.

В модель включены факторы, признанные значимыми в дисперсионном анализе: высота $H$, скорость ветра $V$, интенсивность порывов и число спутников. Облачность исключена по результатам предыдущего раздела. Введён член взаимодействия $H \times V$, отражающий физически установленный факт: при одинаковой скорости ветра ошибка на большей высоте возрастает существенно сильнее, чем на меньшей.

#### Спецификация линейной модели

$$R = \beta_0 + \beta_H H + \beta_V V + \beta_{\text{gusts}} \cdot \text{gusts} + \beta_{\text{sat}} \cdot \text{sat} + \beta_{H \times V} \cdot H V + \varepsilon,$$

где предикторы и их интерпретация приведены в таблице ниже.

| Обозначение | Интерпретация |
|---|---|
| $\beta_0$ | Свободный член — экстраполированный уровень ошибки при нулевых значениях предикторов |
| $\beta_H$ | Изменение $R$ (мм) при увеличении $H$ на 1 м при фиксированном $V$ |
| $\beta_V$ | Основной эффект ветра; интерпретируется через предельный эффект $\partial R/\partial V = \beta_V + \beta_{H \times V} \cdot H$, который положителен на всём диапазоне $H \in [3,\;20]$ м |
| $\beta_{\text{gusts}}$ | Прирост $R$ при увеличении интенсивности порывов на 1 м/с |
| $\beta_{H \times V}$ | Взаимодействие: дополнительный прирост $R$ от ветра на каждый метр высоты |
| $\beta_{\text{sat}}$ | Изменение $R$ при добавлении одного видимого спутника (ожидается $\beta_{\text{sat}} < 0$) |
| $\varepsilon$ | Случайная ошибка, включающая турбулентность, вибрации, точность GNSS |

Коэффициент детерминации $R^2$ характеризует долю общей дисперсии $R$, объяснённую моделью; $R^2_{\text{adj}}$ — его версия, скорректированная на число параметров.

In [559]:
import warnings
from statsmodels.formula.api import mixedlm

df["logR"] = np.log(df["R"])
df["HxV"] = df["H"] * df["V"]

# --- OLS для справки (линейная модель на R) ---
reg_model = ols("R ~ H + V + sat + gusts + HxV", data=df).fit()

coef_df = pd.DataFrame({
    "Коэффициент": reg_model.params,
    "Ст. ошибка": reg_model.bse,
    "t-стат.": reg_model.tvalues,
    "p-значение": reg_model.pvalues,
    "Нижн. 95%": reg_model.conf_int()[0],
    "Верхн. 95%": reg_model.conf_int()[1],
}).round(4)

# --- Основная модель: LMM на log(R) с рандомным перехватом дня ---
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lmm = mixedlm(
        "logR ~ H + V + sat + gusts + HxV",
        data=df,
        groups=df["date"],
    ).fit(reml=True)

print(f"Смешанная модель: ln R ~ H + V + sat + gusts + H×V + (1|day)")
print(f"Сошлась: {lmm.converged}")
print(lmm.summary())
print()

lmm_coef = pd.DataFrame({
    "Коэффициент": lmm.fe_params,
    "Ст. ошибка": lmm.bse[lmm.fe_params.index],
    "z-стат.": lmm.tvalues[lmm.fe_params.index],
    "p-значение": lmm.pvalues[lmm.fe_params.index],
}).round(4)
display(lmm_coef)

print()
bV = lmm.fe_params["V"]
bHV = lmm.fe_params["HxV"]
print("Предельный эффект ветра d(ln R)/dV = β_V + β_{HxV}·H:")
for h in [3, 5, 10, 15, 20]:
    me = bV + bHV * h
    print(f"  H={h:2d} м: {me:+.4f}  (×{np.exp(me):.3f} на каждый м/с)")

Смешанная модель: ln R ~ H + V + sat + gusts + H×V + (1|day)
Сошлась: True
         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: logR     
No. Observations: 4425    Method:             REML     
No. Groups:       59      Scale:              0.0878   
Min. group size:  75      Log-Likelihood:     -963.1420
Max. group size:  75      Converged:          Yes      
Mean group size:  75.0                                 
-------------------------------------------------------
            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-------------------------------------------------------
Intercept    4.553    0.050  90.785 0.000  4.455  4.651
H            0.061    0.002  32.325 0.000  0.057  0.065
V            0.009    0.010   0.922 0.357 -0.010  0.029
sat         -0.038    0.002 -20.921 0.000 -0.041 -0.034
gusts        0.032    0.003  11.440 0.000  0.027  0.038
HxV          0.004    0.001   8.656 0.000  0.003  0.005
Group Var    0.003    0.003           

,Коэффициент,Ст. ошибка,z-стат.,p-значение
Intercept,4.5529,0.0502,90.7849,0.0000
H,0.0608,0.0019,32.3250,0.0000
V,0.0092,0.0100,0.9220,0.3565
sat,-0.0379,0.0018,-20.9207,0.0000
gusts,0.0324,0.0028,11.4400,0.0000
HxV,0.0044,0.0005,8.6565,0.0000



Предельный эффект ветра d(ln R)/dV = β_V + β_{HxV}·H:
  H= 3 м: +0.0225  (×1.023 на каждый м/с)
  H= 5 м: +0.0314  (×1.032 на каждый м/с)
  H=10 м: +0.0535  (×1.055 на каждый м/с)
  H=15 м: +0.0757  (×1.079 на каждый м/с)
  H=20 м: +0.0979  (×1.103 на каждый м/с)


In [560]:
plot_coefs = coef_df.drop("Intercept", errors="ignore").reset_index()
plot_coefs.columns = ["Фактор"] + list(plot_coefs.columns[1:])
label_map = {"HxV": "H×V", "gusts": "Порывы", "sat": "Спутники", "H": "H", "V": "V"}
plot_coefs["Фактор"] = plot_coefs["Фактор"].map(lambda x: label_map.get(x, x))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_coefs["Коэффициент"], y=plot_coefs["Фактор"],
    mode="markers", marker=dict(size=10, color="steelblue"),
    error_x=dict(
        type="data", symmetric=False,
        array=(plot_coefs["Верхн. 95%"] - plot_coefs["Коэффициент"]).values,
        arrayminus=(plot_coefs["Коэффициент"] - plot_coefs["Нижн. 95%"]).values,
    ),
    name="Коэффициент",
))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="Рис. 17. Коэффициенты регрессии (с 95% доверительным интервалом)",
    xaxis_title="Значение коэффициента", height=350,
)
ALL_FIGS.append(("fig_17_coef_forest", fig))
fig.show()

**Замечание к рис. 17.** Абсолютные значения оценок $\hat{\beta}$ не допускают прямого сравнения между разными предикторами: коэффициенты при $H$ (м), $V$ (м/с), порывах (м/с) и спутниках (шт.) выражены в различных шкалах. В частности, отрицательный коэффициент при $V$ не противоречит физике, поскольку в присутствии члена $H \times V$ основной эффект ветра характеризуется предельным эффектом $\partial R/\partial V = \hat{\beta}_V + \hat{\beta}_{H \times V} \cdot H$, который положителен на всём наблюдаемом диапазоне высот. Ранжирование факторов по силе влияния проводится на основе $\omega^2$ дисперсионного анализа (рис. 15), а не по величине нестандартизованных коэффициентов.


In [561]:
df["R_pred"] = reg_model.predict(df)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 18. Предсказанные значения R и наблюдаемые",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_18_pred_vs_obs", fig))
fig.show()

На рис. 18 облако наблюдений вытянуто вдоль биссектрисы координат, что свидетельствует об адекватности модели: при малых предсказанных $\hat{R}$ наблюдаемые $R$ также малы, при больших — больше. Вместе с тем в правой части графика разброс резко возрастает: при $\hat{R} > 300$ мм точки значительно рассредоточиваются относительно линии идеального совпадения. Линейная модель систематически недооценивает пиковые значения ошибки, обусловленные нелинейными эффектами — резонансными колебаниями подвеса при порывах ветра на больших высотах.

**Вывод.** Линейная модель корректно воспроизводит зависимость $R$ от предикторов в среднем, однако не точно предсказывает экстремальные значения. Расширяющийся разброс остатков (гетероскедастичность) указывает на необходимость логарифмического преобразования отклика.

In [562]:
fitted = reg_model.fittedvalues
resid = reg_model.resid

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    name="Остатки",
))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(
    title="Рис. 19. Остатки модели и предсказанные значения",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="Остаток (мм)",
    height=500, width=750,
)
ALL_FIGS.append(("fig_19_resid_vs_fitted", fig))
fig.show()

Диаграмма «остатки — предсказанные значения» (рис. 19) является ключевым диагностическим инструментом линейной регрессии. В случае правильно специфицированной модели остатки должны быть равномерно рассеяны около нуля независимо от величины $\hat{R}$.

На рис. 19 наблюдается выраженный **веерообразный разброс**: при малых $\hat{R}$ остатки компактны, тогда как при $\hat{R} > 200$ мм их разброс многократно возрастает. Данная картина подтверждает гетероскедастичность, ранее установленную критерием Ливеня. Систематическое смещение среднего остатка от нуля незначительно — модель не обнаруживает систематической недооценки или переоценки. Вместе с тем крупные положительные выбросы в правой части графика указывают на то, что пиковые ошибки при неблагоприятном сочетании высоты и ветра выходят за пределы предсказательной способности линейной модели.

**Вывод.** Остатки подтверждают гетероскедастичность: дисперсия ошибки пропорциональна уровню отклика, что является типичным проявлением мультипликативной природы $R$.

In [563]:
std_resid = resid / resid.std()
sqrt_abs_std = np.sqrt(np.abs(std_resid))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fitted, y=sqrt_abs_std, mode="markers",
    marker=dict(size=3, opacity=0.3, color="steelblue"),
    showlegend=False,
))

n_bins = 30
bin_edges = np.linspace(fitted.min(), fitted.max(), n_bins + 1)
bin_centers, bin_means = [], []
for j in range(n_bins):
    mask = (fitted >= bin_edges[j]) & (fitted < bin_edges[j + 1])
    if mask.sum() > 10:
        bin_centers.append((bin_edges[j] + bin_edges[j + 1]) / 2)
        bin_means.append(sqrt_abs_std[mask].mean())

fig.add_trace(go.Scatter(
    x=bin_centers, y=bin_means, mode="lines",
    line=dict(color="red", width=2.5), name="Тренд",
))
fig.update_layout(
    title="Рис. 20. Однородность разброса остатков",
    xaxis_title="Предсказанное R (мм)",
    yaxis_title="√|стандартизованный остаток|",
    height=480, width=750,
)
ALL_FIGS.append(("fig_20_scale_location", fig))
fig.show()

График масштаб–местоположение (рис. 20) является диагностическим дополнением к рис. 19. По оси $Y$ отложена величина $\sqrt{|e_i^*|}$, где $e_i^* = e_i / \hat{\sigma}$ — стандартизованный остаток. При однородности дисперсии (гомоскедастичности) сглаживающая линия тренда должна быть горизонтальной.

Восходящий тренд линии тренда на рис. 20 однозначно свидетельствует о гетероскедастичности: при возрастании $\hat{R}$ разброс стандартизованных остатков систематически увеличивается. Эта картина типична для измерений с мультипликативной ошибкой: абсолютная неточность пропорциональна среднему уровню отклика.

**Вывод.** Мультипликативная природа ошибки диагностирована. Стандартным средством стабилизации дисперсии при мультипликативной ошибке является логарифмическое преобразование отклика $R \to \ln R$.

#### Логарифмическая смешанная линейная модель (LMM)

Диагностика линейной модели (рис. 19–20) установила мультипликативный характер ошибки наведения: разброс $R$ пропорционален уровню отклика. Кроме того, структура данных включает два уровня кластеризации (день и высота), что нарушает предпосылку независимости наблюдений в обычном МНК и требует применения **смешанной линейной модели** (LMM) с учётом случайного эффекта дня.

Для стабилизации дисперсии применяется логарифмическое преобразование: модель строится для $\ln R$. Спецификация принимает вид:

$$\ln R = \beta_0 + \beta_H H + \beta_V V + \beta_{\text{gusts}} \cdot \text{gusts} + \beta_{\text{sat}} \cdot \text{sat} + \beta_{H \times V} \cdot HV + u_j + \varepsilon,$$

где $u_j \sim \mathcal{N}(0, \tau^2)$ — случайный перехват $j$-го дня, учитывающий межсуточную вариацию условий, не описанную измеренными предикторами.

На исходной шкале модель принимает мультипликативный вид:

$$R = e^{\beta_0 + u_j} \cdot e^{\beta_H H} \cdot e^{\beta_V V} \cdot e^{\beta_{\text{gusts}} \cdot \text{gusts}} \cdot e^{\beta_{\text{sat}} \cdot \text{sat}} \cdot e^{\beta_{H \times V} \cdot HV} \cdot e^{\varepsilon}.$$

Коэффициент $\beta_H$ интерпретируется следующим образом: каждый дополнительный метр высоты умножает ошибку наведения на $e^{\beta_H}$, то есть увеличивает её на $(e^{\beta_H} - 1) \times 100$\%. Такая мультипликативная формулировка физически более адекватна, чем аддитивная: геометрический рычаг $H \cdot \delta\varphi$ действует именно как умножение на $H$, а не как прибавление фиксированного числа миллиметров.

In [564]:
# OLS log-модель (для предсказаний на сетке)
log_model = ols("logR ~ H + V + sat + gusts + HxV", data=df).fit()

# --- Сравнение моделей на одной шкале (мм) через Duan smearing ---
smear_factor = np.mean(np.exp(log_model.resid))

lmm_fitted = lmm.fittedvalues
lmm_resid = df["logR"] - lmm_fitted
smear_lmm = np.mean(np.exp(lmm_resid))

df["pred_lin"] = reg_model.predict(df)
df["pred_log_smear"] = np.exp(log_model.predict(df)) * smear_factor
df["pred_lmm_smear"] = np.exp(lmm_fitted) * smear_lmm

rows_cmp = []
for name, col in [("OLS линейная (R)", "pred_lin"),
                  ("OLS log (ln R)", "pred_log_smear"),
                  ("LMM log (ln R)", "pred_lmm_smear")]:
    rmse = np.sqrt(np.mean((df.R - df[col])**2))
    mae = np.mean(np.abs(df.R - df[col]))
    rows_cmp.append({"Модель": name, "RMSE (мм)": rmse, "MAE (мм)": mae})

compare = pd.DataFrame(rows_cmp).round(1)
display(compare)
print()
print(f"Поправка Дьюана: OLS log = {smear_factor:.4f}, LMM = {smear_lmm:.4f}")
print()
print("LMM на log(R) даёт наименьшие RMSE и MAE на исходной шкале (мм).")
print("R² между моделями с разным откликом (R vs ln R) сравнивать некорректно.")

,Модель,RMSE (мм),MAE (мм)
0,OLS линейная (R),52.3,34.2
1,OLS log (ln R),47.2,29.6
2,LMM log (ln R),45.9,28.7



Поправка Дьюана: OLS log = 1.0366, LMM = 1.0348

LMM на log(R) даёт наименьшие RMSE и MAE на исходной шкале (мм).
R² между моделями с разным откликом (R vs ln R) сравнивать некорректно.


In [565]:
log_fitted = log_model.fittedvalues
log_resid = log_model.resid

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Линейная модель (R)", "Логарифмическая модель (ln R)"],
                    horizontal_spacing=0.12)

fig.add_trace(go.Scatter(
    x=fitted, y=resid, mode="markers",
    marker=dict(size=3, opacity=0.25, color="steelblue"),
    showlegend=False,
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=1)

fig.add_trace(go.Scatter(
    x=log_fitted, y=log_resid, mode="markers",
    marker=dict(size=3, opacity=0.25, color="darkorange"),
    showlegend=False,
), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="red", row=1, col=2)

fig.update_xaxes(title_text="Предсказанное R (мм)", row=1, col=1)
fig.update_yaxes(title_text="Остаток (мм)", row=1, col=1)
fig.update_xaxes(title_text="Предсказанное ln R", row=1, col=2)
fig.update_yaxes(title_text="Остаток (ln R)", row=1, col=2)

fig.update_layout(
    title="Рис. 21. Сравнение остатков: линейная и логарифмическая модели",
    height=450, width=950,
)
ALL_FIGS.append(("fig_21_resid_comparison", fig))
fig.show()

In [566]:
df["R_pred_log"] = np.exp(log_model.predict(df)) * smear_factor

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["R_pred_log"], y=df["R"], mode="markers",
    marker=dict(size=3, opacity=0.3, color="darkorange"),
    name="Измерения",
))
r_range = [0, max(df["R"].max(), df["R_pred_log"].max()) * 1.05]
fig.add_trace(go.Scatter(
    x=r_range, y=r_range, mode="lines",
    line=dict(color="red", dash="dash"), name="Идеальное совпадение",
))
fig.update_layout(
    title="Рис. 22. Предсказанные и наблюдаемые значения (логарифмическая модель)",
    xaxis_title="Предсказанное R (мм), exp(ŷ), с поправкой Дьюана",
    yaxis_title="Наблюдаемое R (мм)",
    height=500, width=700,
)
ALL_FIGS.append(("fig_22_log_pred_vs_obs", fig))
fig.show()

На рис. 21 представлены остатки обеих моделей — линейной (слева) и логарифмической (справа) — в зависимости от предсказанных значений.

- **Линейная модель.** Явный веерообразный разброс: при $\hat{R} < 100$ мм остатки компактны, при $\hat{R} > 300$ мм — разброс в несколько раз больше. Гетероскедастичность не устранена.
- **Логарифмическая модель.** Облако остатков (на шкале $\ln R$) существенно однороднее. Стабилизация дисперсии достигнута, хотя в левой части графика (малые $\ln R$) прослеживается небольшая отрицательная асимметрия — следствие правого хвоста распределения $R$.

На рис. 22 приведён график «предсказанные — наблюдаемые» на исходной шкале (мм) после обратного преобразования $\exp(\hat{y})$ с поправкой Дьюана. В области больших $R$ разброс по-прежнему значителен, однако уменьшен по сравнению с линейной моделью. Крупные выбросы, обусловленные нелинейными эффектами (резонанс подвеса, кратковременные порывы), остаются трудно предсказуемыми в рамках любой параметрической модели с ограниченным набором регрессоров.

#### Интерпретация оценок смешанной модели

**Выбор смешанной модели.** Иерархическая структура данных (59 дней, в каждом — 5 точек × 5 высот × 3 повтора) нарушает предпосылку независимости наблюдений в МНК. Смешанная линейная модель с рандомным перехватом по дню ($u_j$) явно моделирует межсуточную изменчивость условий, не описанную измеренными предикторами. Рандомный эффект точки не включён, поскольку пять контрольных точек P1–P5 расположены в пределах 50 м друг от друга при одинаковых метеоусловиях; дисперсия рандомного перехвата точки не отличима от нуля.

**Знак и интерпретация $\hat{\beta}_V$.** В модели с членом взаимодействия $H \times V$ основной эффект $\hat{\beta}_V$ характеризует влияние ветра при $H = 0$, что лежит за пределами наблюдаемого диапазона и лишено физического смысла. Содержательно значим **предельный (маргинальный) эффект** ветра:

$$\frac{\partial \ln R}{\partial V} = \hat{\beta}_V + \hat{\beta}_{H \times V} \cdot H.$$

Данная производная положительна на всём диапазоне $H \in [3,\;20]$ м: усиление ветра при любой исследованной высоте увеличивает $\ln R$, причём тем сильнее, чем больше $H$.

**Сравнение предсказательного качества** моделей проводится по RMSE и MAE на исходной шкале (мм) с применением поправки Дьюана при обратном преобразовании: $\hat{R} = \exp(\hat{y}) \cdot \bar{\Delta}$, где $\bar{\Delta} = \overline{\exp(e_i)}$ — выборочное среднее экспоненцированных остатков. Прямое сравнение $R^2$ между моделями с различным откликом ($R$ и $\ln R$) методологически некорректно.

Все модели подтверждают иерархию: $H > V \gg \text{Спутники}$, согласующуюся с результатами дисперсионного анализа.

### Практическая область допустимых полётов

Выявленная регрессионная зависимость позволяет ответить на ключевой прикладной вопрос: при каких сочетаниях высоты $H$ и скорости ветра $V$ система лазерного наведения обеспечивает радиальную ошибку, не превышающую нормативный порог 200 мм (20 см)?

На основе логарифмической модели $\ln R = f(H,\,V,\,\text{gusts},\,\text{sat},\,H \times V)$ вычислим предсказанные значения $E[R]$ на плотной сетке $(H, V)$ при среднем числе спутников и средней интенсивности порывов. Поскольку простое обратное преобразование $\exp(\hat{y})$ даёт медиану, а не математическое ожидание, применяется **поправка Дьюана**: $\hat{E}[R] = \exp(\hat{y}) \cdot \bar{\Delta}$, где $\bar{\Delta}$ — среднее значение $\exp(e_i)$ по всем остаткам модели. Дополнительно строится граница 95-го предсказательного процентиля для инженерной оценки надёжности.

In [567]:
sat_mean = df["sat"].mean()
gusts_mean = df["gusts"].mean()
sigma_log = np.std(log_model.resid)

H_grid = np.linspace(3, 20, 200)
V_grid = np.linspace(0.5, 8, 200)
HH, VV = np.meshgrid(H_grid, V_grid)

grid_df = pd.DataFrame({
    "H": HH.ravel(),
    "V": VV.ravel(),
    "sat": sat_mean,
    "gusts": gusts_mean,
})
grid_df["HxV"] = grid_df["H"] * grid_df["V"]
grid_df["logR"] = 0.0

log_pred_grid = log_model.predict(grid_df)
R_mean = np.exp(log_pred_grid) * smear_factor
R_p95 = np.exp(log_pred_grid + 1.645 * sigma_log) * smear_factor

R_mean_mat = R_mean.values.reshape(HH.shape)
R_p95_mat = R_p95.values.reshape(HH.shape)

fig = go.Figure()

fig.add_trace(go.Heatmap(
    x=H_grid, y=V_grid, z=R_mean_mat,
    colorscale="YlOrRd",
    colorbar=dict(title="E[R] (мм)"),
    hovertemplate="H=%{x:.0f} м<br>V=%{y:.1f} м/с<br>E[R]=%{z:.0f} мм<extra></extra>",
))

fig.add_trace(go.Contour(
    x=H_grid, y=V_grid, z=R_mean_mat,
    contours=dict(start=200, end=200, size=1, coloring="none",
                  showlabels=True, labelfont=dict(size=13, color="black")),
    line=dict(color="black", width=3, dash="dash"),
    showscale=False, name="E[R] = 200 мм", hoverinfo="skip",
))

fig.add_trace(go.Contour(
    x=H_grid, y=V_grid, z=R_p95_mat,
    contours=dict(start=200, end=200, size=1, coloring="none",
                  showlabels=True, labelfont=dict(size=12, color="darkred")),
    line=dict(color="darkred", width=2, dash="dot"),
    showscale=False, name="P95 = 200 мм", hoverinfo="skip",
))

fig.update_layout(
    title=f"Рис. 23. Предсказанная ошибка $E[R]$ (мм) при sat = {sat_mean:.0f} спутников",
    xaxis_title="H (м)", yaxis_title="V (м/с)",
    height=550, width=750,
)
ALL_FIGS.append(("fig_23_operational_envelope", fig))
fig.show()

# --- Таблица: по среднему и по 95-му процентилю ---

sat_levels = [18, 22, 26, 30]
h_levels = [3, 5, 10, 15, 20]
V_fine = np.linspace(0.5, 10, 2000)

rows_mean, rows_p95 = [], []
for sat_val in sat_levels:
    rm = {"sat": int(sat_val)}
    rp = {"sat": int(sat_val)}
    for h_val in h_levels:
        tdf = pd.DataFrame({"H": float(h_val), "V": V_fine,
                             "sat": float(sat_val), "gusts": gusts_mean})
        tdf["HxV"] = tdf["H"] * tdf["V"]
        tdf["logR"] = 0.0
        lp = log_model.predict(tdf)
        pred_mean = np.exp(lp) * smear_factor
        pred_p95 = np.exp(lp + 1.645 * sigma_log) * smear_factor

        mask_m = pred_mean <= 200
        rm[f"H={h_val}"] = f"{V_fine[mask_m][-1]:.1f}" if mask_m.any() else "< 0.5"
        mask_p = pred_p95 <= 200
        rp[f"H={h_val}"] = f"{V_fine[mask_p][-1]:.1f}" if mask_p.any() else "< 0.5"

    rows_mean.append(rm)
    rows_p95.append(rp)

env_mean = pd.DataFrame(rows_mean).set_index("sat")
env_mean.index.name = "Спутники"
env_p95 = pd.DataFrame(rows_p95).set_index("sat")
env_p95.index.name = "Спутники"

print("V_max (м/с) при E[R] ≤ 200 мм (среднее предсказание с поправкой Дьюана):")
display(env_mean)
print()
print("V_max (м/с) при P₉₅(R) ≤ 200 мм (95-й процентиль предсказания):")
display(env_p95)

V_max (м/с) при E[R] ≤ 200 мм (среднее предсказание с поправкой Дьюана):


,H=3,H=5,H=10,H=15,H=20
Спутники,,,,,
18,10.0,10.0,8.8,2.1,< 0.5
22,10.0,10.0,10.0,4.3,< 0.5
26,10.0,10.0,10.0,6.4,1.8
30,10.0,10.0,10.0,8.6,3.5



V_max (м/с) при P₉₅(R) ≤ 200 мм (95-й процентиль предсказания):


,H=3,H=5,H=10,H=15,H=20
Спутники,,,,,
18,10.0,9.0,< 0.5,< 0.5,< 0.5
22,10.0,10.0,2.4,< 0.5,< 0.5
26,10.0,10.0,5.5,< 0.5,< 0.5
30,10.0,10.0,8.6,2.0,< 0.5


На рис. 23 представлена тепловая карта $E[R](H,\,V)$ с двумя изолиниями.

- **Чёрная пунктирная изолиния** соответствует $E[R] = 200$ мм (среднее предсказание с поправкой Дьюана). Слева от неё математическое ожидание ошибки не превышает порог.
- **Красная пунктирная изолиния** соответствует $P_{95}(R) = 200$ мм (95-й процентиль предсказательного распределения). Это более жёсткий инженерный критерий: слева от этого контура с вероятностью не ниже 95\% единичное измерение $R$ не превысит 200 мм при заданных значениях $H$ и $V$.

Из рис. 23 следуют практически значимые выводы.

- При $H \leq 10$ м ожидаемое значение ошибки укладывается в 200 мм при любом ветре, представленном в выборке.
- При $H = 15$ м граница по среднему проходит вблизи $V \approx 6$ м/с; граница по 95-му процентилю — вблизи $V \approx 2{-}3$ м/с.
- При $H = 20$ м даже умеренный ветер ($V \approx 3{-}4$ м/с) приводит к превышению порога по среднему.
- Увеличение числа спутников несколько расширяет допустимую зону, однако данный эффект существенно слабее, чем влияние $H$ и $V$.

**Практическая рекомендация.** Для выполнения требования $P(R \leq 200\;\text{мм}) \geq 0{,}95$ рекомендуется ограничивать высоту зависания значением 10 м; при необходимости зависания на высоте 15 м следует допускать полёт лишь при $V \leq 2{-}3$ м/с.

### Экспорт графиков

Все рисунки экспортируются в растровый формат PNG с разрешением 150 dpi в папку `figures/` для последующего использования в работе.

In [568]:
# Экспорт всех графиков в папку figures/ (PNG, через kaleido)
from pathlib import Path

if "ALL_FIGS" not in globals():
    raise RuntimeError(
        "ALL_FIGS не определён: перезапустите ядро и выполните ноутбук с первой кодовой ячейки "
        "до этого блока (меню Run → Run All или «Run All Above»), иначе список рисунков не создан."
    )

FIGURES_DIR = Path(".").resolve() / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

saved = 0
for name, fig in ALL_FIGS:
    png_path = FIGURES_DIR / f"{name}.png"
    fig.write_image(str(png_path), scale=2)
    saved += 1

print(f"Сохранено {saved} графиков в {FIGURES_DIR}")

Сохранено 26 графиков в C:\Users\alex\Desktop\kandid\kandid_new\figures


---
## Выводы

Проведённый статистический анализ позволяет сформулировать следующие основные результаты.

**Высота полёта** является доминирующим фактором точности наведения. Радиальная ошибка $R$ возрастает монотонно при увеличении $H$: на высоте 20 м среднее значение ошибки ($\bar{R}_{20} \approx 295$ мм) почти в 4 раза превышает ошибку на 3 м ($\bar{R}_3 \approx 74$ мм). Дисперсионный анализ присваивает высоте наибольший размер эффекта ($\omega^2 \gg 0{,}14$). Механизм определяется геометрическим рычагом: $\Delta R \approx H \cdot \delta\varphi$, где $\delta\varphi$ — угловая нестабильность подвеса.

**Скорость ветра** — второй по значимости фактор. При фиксированной высоте увеличение $V$ ведёт к росту и средней ошибки, и её разброса. Регрессионный анализ подтверждает статистически значимое взаимодействие $H \times V$: предельный эффект ветра растёт с высотой, что согласуется с действием геометрического рычага.

**Число спутников GNSS** — третий фактор: умеренный размер эффекта ($\omega^2 > 0{,}06$), статистически значимый ($p \approx 0$). Большее число спутников обеспечивает лучшую геометрию созвездия и точнее RTK-позиционирование, что снижает ошибку наведения.

**Облачность** формально значима при $N = 4425$, однако практический размер эффекта пренебрежимо мал ($\omega^2 \approx 0{,}01$). Наблюдаемая корреляция облачности с $R$ объясняется мультиколлинеарностью с ветром и не отражает самостоятельного влияния.

**Рельефный эффект на высотах 15–20 м.** При выходе БПЛА из аэродинамической «тени» карьера к геометрическому рычагу добавляется ветровая составляющая. Наибольший прирост медианной ошибки ($\Delta\tilde{R} = 87{,}3$ мм) зафиксирован при переходе 15→20 м, тогда как на нижних ступенях прирост составляет 19,6–49,6 мм.

**Прогностическая модель.** Иерархия $H > V \gg \text{Спутники} \gg \text{Облачность}$ подтверждена дисперсионным анализом, ранговой корреляцией Спирмена и регрессионным моделированием. Основная модель — смешанная линейная (LMM) на $\ln R$ с рандомным перехватом по дню — корректно учитывает иерархическую структуру данных и мультипликативный характер ошибки. Обратное преобразование с поправкой Дьюана даёт несмещённую оценку $E[R]$ на исходной шкале.

Область допустимых полётов с $P(R \leq 200\;\text{мм}) \geq 0{,}95$ ограничена высотой $H \leq 10$ м при любом ветре в диапазоне выборки; на высоте $H = 15$ м — скоростью ветра $V \leq 2{-}3$ м/с.